# 5. (1.5 points)

Given the following to be used in developing a rate indication effective January 1, 2021:

- All policies are annual.
- Rates are expected to be in effect for one year.
- The selected annual loss trend is 2%.

* a.) (0.5 point) Calculate the loss trend factor applied to losses from accident year 2018.
* b.) (0.5 point) Calculate the loss trend factor applied to losses from policy year 2018.
* c.) (0.5 point) Explain why trending and developing losses do not result in overlapping adjustments.

## Solution

Source: Werner & Modlin, Chapter 6 (Losses and LAE), "Loss Trend Periods" and "Overlap Fallacy: Loss Development and Loss Trend".

The loss trend period runs from the **average loss occurrence date of the experience period** ("trend from") to the **average loss occurrence date of the
period in which the rates will be in effect** ("trend to").

**Trend to.** Annual policies written 1/1/2021 to 12/31/2021 provide coverage through 12/31/2022. The midpoint of that two-year period is
12/31/2021 (the text's convention, which is why its 1/1/2015 example uses 12/31/2015).

**Trend from.**
- Accident year 2018: the midpoint of the year, 7/1/2018.
- Policy year 2018: policies written during 2018 cover accidents over a 24-month period (1/1/2018 to 12/31/2019), so the midpoint is 1/1/2019.

In [1]:
import math
import pandas as pd

annual_trend = 0.02

trend_to = pd.Timestamp('2021-12-31')        # midpoint of 1/1/2021 - 12/31/2022 coverage
trend_from = {
    'Accident year 2018': pd.Timestamp('2018-07-01'),   # midpoint of the accident year
    'Policy year 2018': pd.Timestamp('2019-01-01'),     # midpoint of 1/1/2018 - 12/31/2019 coverage
}

# The text rounds these spans to half-years (e.g. 7/1/2011 to 12/31/2015 is 4.5 years)
trend_period = {k: round((trend_to - v).days / 365.25 * 2) / 2 for k, v in trend_from.items()}
trend_factor = {k: (1 + annual_trend) ** n for k, n in trend_period.items()}

assert trend_period == {'Accident year 2018': 3.5, 'Policy year 2018': 3.0}
pd.DataFrame({'trend period (yrs)': trend_period, 'trend factor': trend_factor})

,trend period (yrs),trend factor
Accident year 2018,3.5,1.071768
Policy year 2018,3.0,1.061208


### a.

Accident year 2018: trend period 3.5 years, so the trend factor is $1.02^{3.5}$ = **1.0718**.

### b.

Policy year 2018: trend period 3.0 years, so the trend factor is $1.02^{3.0}$ = **1.0612**.

### c.

Trending and developing do different jobs, so they do not overlap:

- Historical losses occurred months or years before the period the new rates will cover, and they are usually not fully developed at the time of the analysis.
- **Trending** restates losses that occurred in the past to the cost level expected for similar losses occurring in the future period (inflation and other
  factors), moving the average occurrence date forward.
- **Development** brings the immature losses to their expected ultimate level, moving the settlement date out to when the claims close.

The two together project the expected settlement value of the average historical claim as if it occurred at the average date of the future period and
settled with the same reporting and settlement lag. The text's example: trend covers the 30 months from the historical average occurrence date to the future average occurrence
date, and development covers the 18-month settlement lag after that. The loss development does include inflation that applies after a claim is reported, but that
does not make it an overlap, because the trend factor only covers the period up to the average accident date of the future policy period.

## chainladder-python gaps

**What `chainladder` covers.** `cl.Trend` builds compound trend factors along the origin or valuation axis, stored as `trend_`, and can transform a triangle or
feed `CapeCod`. For an annual triangle, `cl.Trend(0.02, axis='origin')` trends each origin to the latest origin, in whole periods.

In [2]:
import numpy as np
import chainladder as cl

demo = pd.DataFrame({'origin': [f'{y}-01-01' for y in range(2018, 2022)],
                     'valuation': [f'{y}-12-31' for y in range(2018, 2022)],
                     'loss': 100.0})
tri = cl.Triangle(demo, origin='origin', development='valuation', columns='loss', cumulative=True)
trend = cl.Trend(annual_trend, axis='origin').fit(tri)
pd.Series(trend.trend_.values[0, 0, :, 0], index=tri.origin.year.tolist(), name='cl.Trend factor to latest origin')

2018    1.061208
2019    1.040400
2020    1.020000
2021    1.000000
Name: cl.Trend factor to latest origin, dtype: float64

This gives 1.0612 for 2018, which is the right answer for part b only if the experience is on a policy-year basis and the latest origin is the policy year in
which the new rates take effect. It does not give part a (1.0718), and it cannot return half-year trend periods.

**What is missing.**

- No trend period calculation from the actuarial inputs: policy term, how long the rates will be in effect, effective date, and whether the experience is accident-year or policy-year.
  The user has to work out the average occurrence dates and pass the right dates and factors by hand.
- No accident-year vs policy-year distinction (a 0.5-year difference in the trend-from date for annual policies), and no sub-annual trend periods (3.5 years for part a).
- No two-step trending (current trend to the latest data point, then projected trend to the forecast period) with different trend rates.

**Suggested feature.** Add a trend-period option to `Trend` that computes the period from the exposure basis: for example
`cl.Trend(0.02, trend_to='2021-01-01', policy_term=12, rate_duration=12, basis='accident')`, with `basis='policy'` for policy-year data. The fitted `trend_` would carry the fractional
trend period for each origin, and a second `trends=[current, projected]` pair would support two-step trending.